# 11_VECTORIAL · Búsqueda semántica de pescado para compradores

MarAzul Sucre · Parcial Final Corte II de Bases de Datos · CECAR

Los compradores y restaurantes piden el pescado con descripciones culinarias informales, como "pescado blanco de carne firme para ceviche fresco", que no coinciden palabra por palabra con lo que guarda la columna `organoleptic_description` de la tabla `species`. Este cuaderno compara la búsqueda léxica con `LIKE` y la búsqueda por similitud de embeddings, guardando los vectores en ChromaDB.

Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**. Si después de la primera celda Colab pide reiniciar la sesión, acepta y vuelve a ejecutar desde la segunda.

In [ ]:
!pip install -q sentence-transformers==6.2.0 chromadb==1.5.9

## 1. Textos del negocio

Son las descripciones organolépticas de las 16 especies de la base `marazul_sucre`, copiadas de la tabla `species` con sus nombres locales.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import chromadb
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

species = [
    {"species_id": 1, "common_name": "Sierra", "scientific_name": "Scomberomorus brasiliensis", "habitat_type": "MARINO", "local_names": "sierra común", "description": "Carne blanca y firme de sabor suave, con poca espina; muy apreciada para ceviche, posta frita y sudados costeños."},
    {"species_id": 2, "common_name": "Pargo rojo", "scientific_name": "Lutjanus purpureus", "habitat_type": "MARINO", "local_names": "pargo colorado", "description": "Piel rojiza y carne blanca, firme y jugosa; ideal para freír entero en fiestas de playa o preparar al horno."},
    {"species_id": 3, "common_name": "Pargo palmero", "scientific_name": "Lutjanus analis", "habitat_type": "MARINO", "local_names": "pargo criollo", "description": "Carne blanca de textura compacta y sabor delicado; se presta para filetes a la plancha y pescado frito entero."},
    {"species_id": 4, "common_name": "Jurel", "scientific_name": "Caranx hippos", "habitat_type": "MARINO", "local_names": "jurel común", "description": "Carne oscura, grasa y de sabor intenso; rinde en guisos, ahumados y sopas de pescado."},
    {"species_id": 5, "common_name": "Róbalo", "scientific_name": "Centropomus undecimalis", "habitat_type": "ESTUARINO", "local_names": "robalo blanco", "description": "Carne blanca, fina y muy firme, sin olor fuerte; considerada de primera para ceviche y restaurantes de mantel."},
    {"species_id": 6, "common_name": "Lisa", "scientific_name": "Mugil incilis", "habitat_type": "ESTUARINO", "local_names": "lisa de ciénaga", "description": "Carne grasa de sabor marcado, muy usada salpresa o frita en ciénagas y playas; su hueva es muy buscada."},
    {"species_id": 7, "common_name": "Mojarra rayada", "scientific_name": "Eugerres plumieri", "habitat_type": "ESTUARINO", "local_names": "mojarra peña", "description": "Pescado pequeño de carne blanca y suave, típico para freír entero y servir con patacón y arroz con coco."},
    {"species_id": 8, "common_name": "Barracuda", "scientific_name": "Sphyraena barracuda", "habitat_type": "MARINO", "local_names": "picuda", "description": "Carne blanca y magra de textura firme; buena en postas a la parrilla, aunque los ejemplares grandes se evitan por toxinas."},
    {"species_id": 9, "common_name": "Langosta espinosa", "scientific_name": "Panulirus argus", "habitat_type": "MARINO", "local_names": "langosta del Caribe", "description": "Crustáceo de carne blanca, dulce y firme; producto de alto valor para hoteles y restaurantes del golfo."},
    {"species_id": 10, "common_name": "Camarón blanco", "scientific_name": "Litopenaeus schmitti", "habitat_type": "MARINO", "local_names": "camarón blanco del Caribe", "description": "Camarón de carne tierna y dulce; ideal para cócteles, arroces y ceviche de camarón."},
    {"species_id": 11, "common_name": "Bocachico", "scientific_name": "Prochilodus magdalenae", "habitat_type": "DULCEACUICOLA", "local_names": "bocachico de ciénaga", "description": "Pescado de ciénaga con muchas espinas y sabor intenso; tradicional frito, en viudo o sancocho de pescado."},
    {"species_id": 12, "common_name": "Bagre rayado", "scientific_name": "Pseudoplatystoma magdaleniatum", "habitat_type": "DULCEACUICOLA", "local_names": "bagre pintado, pintadillo", "description": "Carne blanca, firme y sin espinas intramusculares; excelente en postas, sudado y viudo de bagre."},
    {"species_id": 13, "common_name": "Moncholo", "scientific_name": "Hoplias malabaricus", "habitat_type": "DULCEACUICOLA", "local_names": "dormilón, guabina", "description": "Carne blanca y sabrosa pero con espinas; se consume frito o en sopa en los pueblos ribereños."},
    {"species_id": 14, "common_name": "Mojarra amarilla", "scientific_name": "Caquetaia kraussii", "habitat_type": "DULCEACUICOLA", "local_names": "mojarra de río", "description": "Carne blanca y suave con piel amarillenta; muy apetecida frita entera en las ciénagas del San Jorge."},
    {"species_id": 15, "common_name": "Cachama blanca", "scientific_name": "Piaractus brachypomus", "habitat_type": "DULCEACUICOLA", "local_names": "paco, pirapitinga", "description": "Pez de cultivo de carne rosada, jugosa y con grasa; ideal para asar a la parrilla o en costillas fritas."},
    {"species_id": 16, "common_name": "Tilapia roja", "scientific_name": "Oreochromis sp.", "habitat_type": "DULCEACUICOLA", "local_names": "mojarra roja", "description": "Pez de cultivo de carne blanca, suave y sin sabor a barro; se vende entero para freír o en filete."}
]

catalog = pd.DataFrame(species)
catalog[["species_id", "common_name", "habitat_type", "description"]]

## 2. Búsqueda léxica con LIKE

Se cargan las mismas descripciones en una tabla SQL temporal y se ejecutan las consultas `LIKE` que haría un sistema tradicional. Cada patrón sale de una petición real de comprador.

In [ ]:
conn = sqlite3.connect(":memory:")
catalog.to_sql("species", conn, index=False)

like_patterns = [
    ("pescado blanco de carne firme para ceviche fresco", "%pescado blanco de carne firme para ceviche fresco%"),
    ("pescado blanco de carne firme para ceviche fresco", "%ceviche%"),
    ("pescado ideal para freír entero en fiesta de playa", "%fiesta de playa%"),
    ("pescado ideal para freír entero en fiesta de playa", "%freír entero%"),
    ("algo para un sancocho de pescado de río", "%sancocho%"),
    ("marisco fino para un restaurante de hotel", "%marisco fino%"),
    ("pescado de cultivo jugoso para asar a la parrilla", "%cultivo jugoso%"),
]

like_rows = []
for request, pattern in like_patterns:
    found = pd.read_sql_query("SELECT common_name FROM species WHERE description LIKE ?", conn, params=(pattern,))
    like_rows.append({
        "petición del comprador": request,
        "patrón LIKE": pattern,
        "resultados": len(found),
        "especies encontradas": ", ".join(found["common_name"]) if len(found) else "ninguna",
    })

like_results = pd.DataFrame(like_rows)
like_results

## 3. Generación de embeddings

Se usa el modelo multilingüe `intfloat/multilingual-e5-base`, que convierte cada texto en un vector de 768 números. Este modelo pide anteponer `passage: ` a los textos guardados y `query: ` a las preguntas. Los vectores se normalizan para comparar por similitud coseno.

In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-base")

passages = [f"passage: {s['common_name']}. {s['description']}" for s in species]
passage_vectors = model.encode(passages, normalize_embeddings=True, convert_to_numpy=True)

print("Textos vectorizados:", passage_vectors.shape[0])
print("Dimensiones de cada vector:", passage_vectors.shape[1])
print("Primeros valores del vector de", species[0]["common_name"], ":", np.round(passage_vectors[0][:8], 4))

## 4. Almacenamiento vectorial en ChromaDB

Los vectores se guardan en una colección persistente de ChromaDB con distancia coseno, junto con el texto original y metadatos que permiten filtrar, como el hábitat.

In [ ]:
client = chromadb.PersistentClient(path="chroma_marazul")

try:
    client.delete_collection("species_profiles")
except Exception:
    pass

collection = client.create_collection(
    name="species_profiles",
    configuration={"hnsw": {"space": "cosine"}},
    embedding_function=None,
)

collection.add(
    ids=[f"species_{s['species_id']}" for s in species],
    embeddings=passage_vectors.tolist(),
    documents=[s["description"] for s in species],
    metadatas=[{"species_id": s["species_id"], "common_name": s["common_name"], "habitat_type": s["habitat_type"]} for s in species],
)

print("Vectores almacenados:", collection.count())
print("Configuración de la colección:", collection.configuration["hnsw"]["space"])

## 5. Consultas de los compradores

Cada petición se convierte en vector con el mismo modelo y ChromaDB devuelve las especies más cercanas. La similitud es `1 - distancia coseno`, así que un valor más alto indica un significado más parecido.

In [ ]:
buyer_requests = [
    "pescado blanco de carne firme para ceviche fresco",
    "pescado ideal para freír entero en fiesta de playa",
    "algo para un sancocho de pescado de río",
    "marisco fino para un restaurante de hotel",
    "pescado de cultivo jugoso para asar a la parrilla",
]

def semantic_search(request, k=3, where=None):
    query_vector = model.encode([f"query: {request}"], normalize_embeddings=True, convert_to_numpy=True)
    result = collection.query(
        query_embeddings=query_vector.tolist(),
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    rows = []
    for rank, (meta, doc, dist) in enumerate(zip(result["metadatas"][0], result["documents"][0], result["distances"][0]), start=1):
        rows.append({
            "petición del comprador": request,
            "puesto": rank,
            "especie": meta["common_name"],
            "hábitat": meta["habitat_type"],
            "similitud": round(1 - dist, 3),
            "descripción": doc,
        })
    return rows

semantic_results = pd.DataFrame([row for request in buyer_requests for row in semantic_search(request)])
semantic_results

In [ ]:
comparison = []
for request in buyer_requests:
    lexical = like_results[like_results["petición del comprador"] == request]
    top = semantic_results[semantic_results["petición del comprador"] == request]
    comparison.append({
        "petición del comprador": request,
        "LIKE (mejor patrón)": lexical["especies encontradas"].iloc[-1] if len(lexical) else "ninguna",
        "búsqueda semántica (top 3)": ", ".join(f"{e} ({s})" for e, s in zip(top["especie"], top["similitud"])),
    })

pd.DataFrame(comparison)

## 6. Búsqueda semántica con filtro de metadatos

La búsqueda por significado se puede combinar con filtros exactos. Aquí se restringe la petición del sancocho a especies de agua dulce.

In [ ]:
pd.DataFrame(semantic_search("algo para un sancocho de pescado de río", k=3, where={"habitat_type": "DULCEACUICOLA"}))

## 7. Mapa de similitud entre peticiones y especies

Cada celda muestra qué tan parecido es el significado de una petición con la descripción de cada especie.

In [ ]:
query_vectors = model.encode([f"query: {r}" for r in buyer_requests], normalize_embeddings=True, convert_to_numpy=True)
similarity = query_vectors @ passage_vectors.T

fig, ax = plt.subplots(figsize=(14, 4.5))
image = ax.imshow(similarity, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(species)))
ax.set_xticklabels([s["common_name"] for s in species], rotation=60, ha="right")
ax.set_yticks(range(len(buyer_requests)))
ax.set_yticklabels(buyer_requests)
for i in range(similarity.shape[0]):
    for j in range(similarity.shape[1]):
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(image, ax=ax, label="similitud coseno")
ax.set_title("Similitud semántica entre peticiones de compradores y especies")
plt.tight_layout()
plt.show()

## 8. Integración con la base relacional

La búsqueda semántica devuelve las especies adecuadas y la base MySQL dice cuánto hay disponible de ellas. Esta celda arma la consulta que se ejecutaría sobre la vista de capturas disponibles.

In [ ]:
best_for_ceviche = semantic_results[semantic_results["petición del comprador"] == buyer_requests[0]]["especie"].tolist()
names = ", ".join(f"'{n}'" for n in best_for_ceviche)
print("SELECT common_species_name, SUM(available_weight_kg) AS kg_disponibles")
print("FROM vw_available_catches_at_dock")
print(f"WHERE common_species_name IN ({names})")
print("GROUP BY common_species_name;")